[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/21_dimensionality_reduction/90_exercises.ipynb)

# Bài tập — Chương 21: Dimensionality Reduction (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/21_dimensionality_reduction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)
digits = load_digits()

## L0 — Nhớ khái niệm

**Câu 1.** Viết bài toán tối ưu của PCA (thành phần thứ nhất), liên hệ với trị riêng/vector riêng; công thức explained variance ratio; liên hệ SVD ↔ PCA. t-SNE tối thiểu đại lượng gì?

✍️ Trả lời:

**Đáp án:** $\max_{\lVert\mathbf w\rVert = 1}\mathbf w^\top C\mathbf w$ với $C = \frac{1}{n-1}X_c^\top X_c$ → $C\mathbf w = \lambda\mathbf w$, PC1 là vector riêng của trị riêng lớn nhất, phương sai = $\lambda_1$. Explained variance ratio $= \lambda_j/\sum_k\lambda_k$. $X_c = U\Sigma V^\top$: thành phần chính = cột của $V$, $\lambda_j = \sigma_j^2/(n - 1)$. t-SNE tối thiểu $\text{KL}(P\Vert Q)$ giữa phân phối tương đồng ở không gian gốc (Gaussian) và không gian thấp chiều (Student-t).

## L1 — Giải thích bằng lời

**Câu 2.** Một nhà phân tích trình bày biểu đồ t-SNE: "Cụm khách hàng VIP nhỏ và nằm xa hẳn các cụm khác, nên VIP rất khác biệt và đồng nhất". Nhận xét nào đúng, nào không có căn cứ?

✍️ Trả lời:

**Gợi ý:** t-SNE chỉ đáng tin về **láng giềng cục bộ**: việc khách VIP tạo thành một nhóm riêng (các điểm gần nhau) có thể phản ánh cấu trúc thật — nên kiểm chứng bằng cách khác (clustering trên không gian gốc, so sánh feature). Nhưng **kích thước** cụm (nhỏ ≠ đồng nhất — t-SNE có xu hướng làm các cụm có cỡ tương tự) và **khoảng cách giữa các cụm** (xa ≠ rất khác biệt) không có ý nghĩa định lượng; bố cục còn phụ thuộc perplexity và seed. Để đo "khác biệt", so sánh trực tiếp phân phối feature hoặc khoảng cách trong không gian gốc.

## L2 — Tính tay

**Câu 3.** Ma trận hiệp phương sai $C = \begin{bmatrix}4 & 2\\2 & 1\end{bmatrix}$. Tìm trị riêng, vector riêng PC1 (chuẩn hoá), tỉ lệ phương sai PC1 giải thích.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
# STARTER: lam1, lam2 = None, None   # trị riêng (giải det(C − λI) = 0), lam1 ≥ lam2
# STARTER: pc1 = None                # vector riêng chuẩn hoá của lam1
# STARTER: ratio1 = None
lam1, lam2 = 5.0, 0.0                          # det(C − λI) = λ² − 5λ = 0
pc1 = np.array([2, 1]) / np.sqrt(5)            # (C − 5I) w = 0 → −w1 + 2 w2 = 0
ratio1 = lam1 / (lam1 + lam2)
### END CODE HERE ###

In [ ]:
C = np.array([[4, 2], [2, 1.0]])
vals, vecs = np.linalg.eigh(C)
check_close(sorted([lam1, lam2]), vals, tol=1e-9, message="trị riêng")
check_close(np.abs(pc1 @ vecs[:, -1]), 1.0, message="PC1 cùng phương với vector riêng lớn nhất")
check_close(ratio1, 1.0, message="PC1 giải thích 100% (dữ liệu nằm trên một đường thẳng)")

**Câu 4.** Dữ liệu 3D có trị riêng $[6, 3, 1]$. (a) Tỉ lệ phương sai giải thích của 2 thành phần đầu. (b) Sai số tái tạo trung bình (theo phương sai) khi chỉ giữ 1 thành phần. (c) Cần ít nhất bao nhiêu thành phần để giữ ≥ 85% phương sai?

In [ ]:
lam = np.array([6, 3, 1.0])
### START CODE HERE ### (≈ 3 dòng)
ratio_2 = lam[:2].sum() / lam.sum()
recon_err_1 = lam[1:].sum()
k_85 = int(np.searchsorted(np.cumsum(lam) / lam.sum(), 0.85) + 1)
### END CODE HERE ###

In [ ]:
check_close(ratio_2, 0.9, message="2 thành phần giải thích 90%")
check_close(recon_err_1, 4.0, message="sai số = tổng trị riêng bị bỏ")
check_equal(k_85, 2, "cần 2 thành phần cho ≥ 85%")

## L3 — Code

**Bài 5.** Cài đặt `class MyPCA` với `fit` (qua SVD), `transform`, `inverse_transform`, thuộc tính `components_`, `explained_variance_ratio_`. Kiểm tra với sklearn trên digits (so sánh có tính đến dấu ±).

In [ ]:
class MyPCA:
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        ### START CODE HERE ### (≈ 6 dòng)
        # STARTER: # self.mean_; SVD của X − mean; var = S² / (n − 1)
        # STARTER: # self.components_ = n_components hàng đầu của Vᵀ; self.explained_variance_ratio_ = (var / var.sum())[:n_components]
        X = np.asarray(X, float)
        self.mean_ = X.mean(0)
        _, S, Vt = np.linalg.svd(X - self.mean_, full_matrices=False)
        var = S**2 / (len(X) - 1)
        self.components_ = Vt[: self.n_components]
        self.explained_variance_ratio_ = (var / var.sum())[: self.n_components]
        ### END CODE HERE ###
        return self

    def transform(self, X):
        ### START CODE HERE ### (≈ 1 dòng)
        # STARTER: return None   # (X − mean) @ componentsᵀ
        return (np.asarray(X, float) - self.mean_) @ self.components_.T
        ### END CODE HERE ###

    def inverse_transform(self, Z):
        ### START CODE HERE ### (≈ 1 dòng)
        # STARTER: return None   # Z @ components + mean
        return Z @ self.components_ + self.mean_
        ### END CODE HERE ###

In [ ]:
Xd = digits.data
mine = MyPCA(10).fit(Xd)
sk = PCA(10).fit(Xd)
check_close(mine.explained_variance_ratio_, sk.explained_variance_ratio_, tol=1e-8, message="explained variance ratio")
check_close(np.abs((mine.components_ * sk.components_).sum(1)), np.ones(10), tol=1e-6, message="thành phần trùng (đến dấu)")
check_close(mine.inverse_transform(mine.transform(Xd[:5])), sk.inverse_transform(sk.transform(Xd[:5])), tol=1e-6, message="tái tạo khớp sklearn")

**Bài 6.** Viết `n_components_for_variance(X, target)` trả về số thành phần nhỏ nhất đạt phương sai tích luỹ ≥ `target`, và `reconstruction_mse(X, k)` (MSE tái tạo khi giữ $k$ thành phần). Vẽ MSE theo $k$ cho digits.

In [ ]:
def n_components_for_variance(X, target=0.9):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: ratio = None
    # STARTER: return None   # số thành phần nhỏ nhất có cumsum(ratio) ≥ target
    ratio = PCA().fit(X).explained_variance_ratio_
    return int(np.searchsorted(np.cumsum(ratio), target) + 1)
    ### END CODE HERE ###


def reconstruction_mse(X, k):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: p = None
    # STARTER: return None   # MSE giữa X và inverse_transform(transform(X))
    p = PCA(k).fit(X)
    return np.mean((X - p.inverse_transform(p.transform(X))) ** 2)
    ### END CODE HERE ###

In [ ]:
ks = [1, 2, 5, 10, 20, 40]
mses = [reconstruction_mse(Xd, k) for k in ks]
plt.plot(ks, mses, "o-")
plt.xlabel("k")
plt.ylabel("MSE tái tạo")
plt.show()
check_equal(n_components_for_variance(Xd, 0.95), PCA(0.95).fit(Xd).n_components_, "khớp PCA(0.95)")
check_true(all(np.diff(mses) < 0), "MSE giảm khi k tăng")

## L4 — Debug

**Bài 7.** Pipeline sau dùng PCA trước KNN cho dữ liệu khách hàng nhưng (1) PCA fit trên **toàn bộ** dữ liệu trước CV, (2) không chuẩn hoá (thu nhập theo VND áp đảo), (3) số thành phần cố định = 1. Sửa: Pipeline(StandardScaler → PCA → KNN), chọn `n_components` ∈ {1, 2, 3, 4} bằng GridSearchCV (5-fold). Lưu `best_nc` và `cv_acc`.

In [ ]:
seg = pd.read_csv(repo_path("data", "sample", "customer_segments.csv"))
X7 = seg[["annual_income_musd", "spending_score", "age", "visits_per_month"]].assign(annual_income_musd=lambda d: d.annual_income_musd * 1e6)
y7 = seg["true_segment"]
### START CODE HERE ### (≈ 6 dòng)
# STARTER: Z7 = PCA(1).fit_transform(X7)                                                  # 🐛 (1), (2), (3)
# STARTER: cv_acc = cross_val_score(KNeighborsClassifier(5), Z7, y7, cv=5).mean()
# STARTER: best_nc = 1
grid7 = GridSearchCV(make_pipeline(StandardScaler(), PCA(), KNeighborsClassifier(5)), {"pca__n_components": [1, 2, 3, 4]},
                     cv=StratifiedKFold(5, shuffle=True, random_state=0)).fit(X7, y7)
best_nc = grid7.best_params_["pca__n_components"]
cv_acc = grid7.best_score_
### END CODE HERE ###
print(f"n_components = {best_nc} | accuracy CV = {cv_acc:.3f}")

In [ ]:
check_greater(cv_acc, 0.9, "accuracy CV > 0.9")
check_greater(best_nc, 1, "cần hơn 1 thành phần")

## L5 — Phân tích

**Bài 8.** Khảo sát **trustworthiness** (k = 10) của PCA và t-SNE (perplexity ∈ {5, 30, 50}) trên 800 ảnh digits đầu; lưu vào dict `trust`. Thêm thí nghiệm: chạy t-SNE (perplexity 30) với 3 seed khác nhau và đo **khoảng cách giữa tâm chữ số 0 và chữ số 6** trong 2D (`dist_06`, list). Nhận xét về độ ổn định của khoảng cách giữa cụm.

In [ ]:
X8, y8 = digits.data[:800], digits.target[:800]
### START CODE HERE ### (≈ 10 dòng)
# STARTER: trust = None     # dict: "PCA" và "t-SNE p=5/30/50" → trustworthiness(X8, Z, n_neighbors=10)
# STARTER: dist_06 = None   # list 3 khoảng cách giữa tâm chữ số 0 và 6 (t-SNE perplexity 30, seed 0/1/2, bố cục đã chuẩn hoá)
trust = {"PCA": trustworthiness(X8, PCA(2).fit_transform(X8), n_neighbors=10)}
for perp in [5, 30, 50]:
    trust[f"t-SNE p={perp}"] = trustworthiness(X8, TSNE(2, perplexity=perp, random_state=0).fit_transform(X8), n_neighbors=10)
dist_06 = []
for seed in [0, 1, 2]:
    Z8 = TSNE(2, perplexity=30, random_state=seed).fit_transform(X8)
    Zs8 = (Z8 - Z8.mean(0)) / Z8.std(0)                    # chuẩn hoá bố cục để so sánh
    dist_06.append(float(np.linalg.norm(Zs8[y8 == 0].mean(0) - Zs8[y8 == 6].mean(0))))
### END CODE HERE ###
print({k: round(v, 3) for k, v in trust.items()}, "| khoảng cách tâm 0–6 qua các seed:", np.round(dist_06, 2))

In [ ]:
check_greater(min(v for k, v in trust.items() if k.startswith("t-SNE")), trust["PCA"], "t-SNE giữ láng giềng tốt hơn PCA")
check_equal(len(dist_06), 3, "3 seed")

✍️ Nhận xét:

**Gợi ý:** t-SNE có trustworthiness cao hơn PCA nhiều (giữ láng giềng cục bộ tốt). Với `init="pca"` (mặc định từ scikit-learn 1.2), bố cục khá **ổn định** giữa các seed — khoảng cách tâm 0–6 gần như không đổi. Nhưng ổn định ≠ đúng: khoảng cách giữa cụm trên t-SNE vẫn bị méo so với không gian gốc (bài 03, mục 3a — cụm rất xa và cụm gần trông cách đều nhau), và với `init="random"` hoặc perplexity khác, bố cục có thể thay đổi đáng kể. Dùng t-SNE để thấy nhóm, không để đo.

## L6 — Thực tế

**Bài 9.** Nén ảnh khuôn mặt giả lập: tạo 400 "ảnh" 20×20 từ 6 "khuôn mẫu" ngẫu nhiên (mỗi ảnh = tổ hợp tuyến tính ngẫu nhiên của khuôn mẫu + nhiễu nhỏ). Dùng PCA để: (a) tìm số thành phần giữ ≥ 99% phương sai (`k99`); (b) tính tỉ lệ nén (số con số lưu trữ khi giữ `k99` thành phần, gồm cả các thành phần và mean, so với dữ liệu gốc) — `compression`; (c) MSE tái tạo tương đối (`rel_mse` = MSE / phương sai dữ liệu).

In [ ]:
templates = rng.normal(size=(6, 400))
coef = rng.normal(size=(400, 6))
faces = coef @ templates + rng.normal(0, 0.1, (400, 400))
### START CODE HERE ### (≈ 6 dòng)
p9 = PCA(0.99).fit(faces)
k99 = p9.n_components_
stored = 400 * k99 + k99 * 400 + 400                      # toạ độ mỗi ảnh + các thành phần + mean
compression = stored / faces.size
rel_mse = np.mean((faces - p9.inverse_transform(p9.transform(faces))) ** 2) / faces.var()
### END CODE HERE ###
print(f"k99 = {k99} | lưu {compression:.1%} dữ liệu gốc | MSE tương đối = {rel_mse:.4f}")

In [ ]:
check_less(k99, 10, "cấu trúc 6 khuôn mẫu → cần rất ít thành phần")
check_less(compression, 0.1, "nén còn < 10%")
check_less(rel_mse, 0.01, "sai số tái tạo < 1% phương sai")

## L7 — Nâng cao

**Bài 10.** **Power iteration**: tìm PC1 không cần SVD đầy đủ — lặp $\mathbf w \leftarrow C\mathbf w / \lVert C\mathbf w\rVert$ (hội tụ về vector riêng trội). Sau đó **deflation**: $C \leftarrow C - \lambda_1\mathbf w_1\mathbf w_1^\top$ để tìm PC2. Cài đặt `top_k_components(X, k, n_iter)` và so với sklearn trên digits (k = 3).

In [ ]:
def top_k_components(X, k=3, n_iter=200, seed=0):
    r = np.random.default_rng(seed)
    Xc = X - X.mean(0)
    C = Xc.T @ Xc / (len(X) - 1)
    comps, vals = [], []
    ### START CODE HERE ### (≈ 8 dòng)
    # STARTER: # k lần: w ngẫu nhiên; lặp n_iter: w = C @ w rồi chuẩn hoá; lam = wᵀ C w; lưu w, lam
    # STARTER: # deflation: C = C − lam · w wᵀ
    for _ in range(k):
        w = r.normal(size=C.shape[0])
        for _ in range(n_iter):
            w = C @ w
            w /= np.linalg.norm(w)
        lam = w @ C @ w
        comps.append(w)
        vals.append(lam)
        C = C - lam * np.outer(w, w)
    ### END CODE HERE ###
    return np.array(comps), np.array(vals)

In [ ]:
comps10, vals10 = top_k_components(Xd, 3)
sk10 = PCA(3).fit(Xd)
check_close(vals10, sk10.explained_variance_, tol=1e-4, message="trị riêng khớp sklearn")
check_close(np.abs((comps10 * sk10.components_).sum(1)), np.ones(3), tol=1e-4, message="vector riêng khớp (đến dấu)")